# 🏏 Notebook 1: Cricinfo — Class Design


## 🛠️ Setup

```bash
cd 07-object-oriented-design/cricinfo
uv sync
```

Select the `.venv` kernel in VS Code (top-right of the notebook). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


## 🎯 What you'll learn

We'll design **Cricinfo** — a live cricket score tracker — step by step. We'll move from a **bad** all-in-one-dict script to a **clean** object-oriented design.

Cricket is a great OOD problem because:
- It has lots of **vocabulary** (teams, players, overs, balls, wickets, extras) that maps naturally to classes.
- It has **rules** (1 over = 6 legal balls, innings ends at 10 wickets or over limit) that are easy to get wrong if logic is scattered.
- It has **two symmetric sides** (batting / bowling) — perfect for thinking about responsibilities.

> 💡 **Never watched cricket?** Two teams take turns batting. The batting side scores **runs** one ball at a time. The bowling side tries to get **wickets** (outs). An **over** is 6 legal balls. After a set number of overs (or 10 wickets), the sides swap. Whoever scores more runs wins.


## 🚫 Bad design: one big function juggling dicts

Before we reach for classes, let's see what happens when we just track everything in a dict. It works for the happy path, but every new rule tangles this function more.


In [ ]:
# A "just make it work" ball-by-ball score tracker.
# No classes, lots of hard-coded strings, rules mixed with state.

def score_innings_badly(balls):
    state = {"runs": 0, "wickets": 0, "legal_balls": 0}
    for b in balls:
        # b is a tuple: (runs, type, wicket?)
        runs, btype, wicket = b
        state["runs"] += runs
        if btype in ("wide", "no_ball"):
            state["runs"] += 1          # +1 penalty run
            # wides/no-balls DO NOT count as legal balls — easy to forget!
        else:
            state["legal_balls"] += 1
        if wicket:
            state["wickets"] += 1
        if state["wickets"] >= 10 or state["legal_balls"] >= 20 * 6:
            break
    overs = f"{state['legal_balls']//6}.{state['legal_balls']%6}"
    return f"{state['runs']}/{state['wickets']} ({overs} ov)"

demo = [
    (1, "legal",  False),
    (4, "legal",  False),
    (0, "legal",  True),        # wicket
    (2, "wide",   False),       # 2 runs taken off a wide (+1 penalty = 3)
    (6, "legal",  False),
    (0, "legal",  False),
    (1, "legal",  False),
]
print(score_innings_badly(demo))


### Why this is "bad"

- **Strings as types** (`"wide"`, `"no_ball"`). A typo (`"noball"`) silently breaks scoring.
- **Rules scattered everywhere**: the "+1 for extras", "don't increment legal_balls for wides", and "end at 10 wickets" live in the same loop. Adding *one* more rule (byes, powerplay, DLS) means editing this function again.
- **No reuse**: a second innings would copy-paste this whole thing.
- **No testability**: to test "what happens on the last ball of the over?" you have to build an entire innings.

The fix is classic OOD: give each **noun** in the domain its own class with clear **responsibilities**.


## ✅ Better design: one class per domain noun

Here's the hierarchy we're going to build:

```
 Team ───▶ Player
  *           (each team has many players)
  │
  └──── Match ────┐ (match has exactly 2 teams)
           │
           │ 1..2
           ▼
        Innings ───▶ Over ───▶ Ball
        (batting side scores over-by-over, ball-by-ball)
```

### Key rules to encode once, in the right class

| Rule | Lives in |
|---|---|
| "A wide adds 1 run but is not a legal ball" | `Ball.total_runs()` / `Ball.is_legal()` |
| "1 over = 6 legal balls" | `Over` / `Innings.over_count()` |
| "Innings ends at 10 wickets OR over limit" | `Innings.is_complete()` |
| "Higher total wins" | `Match.winner()` |

### Class responsibilities

| Class | Responsibility | Doesn't care about |
|---|---|---|
| `Player` | name, role | scoring |
| `Team` | list of players, team name | ball-by-ball rules |
| `Ball` | did this ball count? how many runs? wicket? | who bowled / faced |
| `Over` | up to 6 legal balls | match totals |
| `Innings` | runs, wickets, overs for one batting side | the other innings |
| `Match` | 2 teams, 2 innings, winner | single-ball mechanics |

**Single Responsibility Principle**: if a bug is about extras, you fix `Ball`. If it's about when an innings ends, you fix `Innings`. You never have to hunt through one giant function.


## 🧱 Enums first — no more magic strings

In the bad version, `btype == "wide"` was a string comparison waiting to break. An `Enum` makes the set of valid values explicit and catches typos at write-time instead of silently at run-time.


In [ ]:
from enum import Enum

class BallType(Enum):
    LEGAL   = "legal"
    WIDE    = "wide"     # +1 penalty run, does NOT count as a legal ball
    NO_BALL = "no_ball"  # +1 penalty run, does NOT count as a legal ball

# ✅ Safe: autocomplete + typo caught by Python
print(BallType.WIDE)
print(BallType.WIDE != BallType.LEGAL)

# ❌ In the bad version, this typo would silently be treated as a legal ball:
#     btype = "wde"
# With the Enum, BallType("wde") raises ValueError immediately.
try:
    BallType("wde")
except ValueError as e:
    print("Enum caught the typo:", e)


## 🧠 Design checklist before we code

Before jumping to Notebook 2, convince yourself that:

1. **Every noun has a home.** *"Where does the 'did this ball count?' logic live?"* → `Ball.is_legal()`.
2. **No class reaches into another's internals.** `Match` asks `Innings` for its summary; it doesn't read `innings.legal_balls` directly to compute overs.
3. **Each class can be tested alone.** We can write `assert Ball(runs=2, ball_type=BallType.WIDE).total_runs() == 3` without building a whole match.
4. **Adding a new rule touches one class.** Byes? Extend `Ball`. Super over? Extend `Match`.

👉 In Notebook 2 we implement this design and run a full toy match.
